# Replication Workflow

Single working notebook for the Segnon and Trede replication. Keep durable logic in `src/`; use this notebook for exploration, diagnostics, and figures.

In [116]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys

import numpy as np
import pandas as pd
import plotly.graph_objects as go

PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import (
    RAW_DATA_DIR,
    PROCESSED_DATA_DIR,
    REPORTS_DIR,
    TABLES_DIR,
    ensure_project_dirs,

    align_return_frame,
    load_price_csv,
    log_returns,

    plot_price_evolution,
    plot_returns_and_squared_returns,
    plot_var_forecasts,

    summary_statistics,
    format_table_1,

    fit_msm,
    fit_msm_grid,
    build_msm_pit_frame,
    msm_filter_from_result,

    fit_garch_marginals,
    garch_results_table,
    format_garch_table_3,
    build_garch_pit_frame,
    build_garch_volatility_frame,

    fit_copula_grid,
    copula_results_table,
    format_copula_table_4,

    forecast_msm_copula_var_oos_fixed_params,

    portfolio_returns,
    var_exceedances,
    violation_rate,
    kupiec_pof_test,
    christoffersen_lr_test,
)

ensure_project_dirs()

RAW_DATA_DIR, PROCESSED_DATA_DIR, REPORTS_DIR, TABLES_DIR

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


(WindowsPath('C:/Users/alixg/OneDrive - Université Paris-Dauphine/M2 Quant/Gestion Quantitative/Réplication 2/Forecasting_market_risk-Copula_MSM_Approach/data/raw'),
 WindowsPath('C:/Users/alixg/OneDrive - Université Paris-Dauphine/M2 Quant/Gestion Quantitative/Réplication 2/Forecasting_market_risk-Copula_MSM_Approach/data/processed'),
 WindowsPath('C:/Users/alixg/OneDrive - Université Paris-Dauphine/M2 Quant/Gestion Quantitative/Réplication 2/Forecasting_market_risk-Copula_MSM_Approach/reports'),
 WindowsPath('C:/Users/alixg/OneDrive - Université Paris-Dauphine/M2 Quant/Gestion Quantitative/Réplication 2/Forecasting_market_risk-Copula_MSM_Approach/reports/tables'))

## 1. Prices

In [90]:
raw_data_names = ["nasdaqcom_yahoo_close.csv", "sp500_yahoo_close.csv"]
raw_data_paths = [RAW_DATA_DIR / name for name in raw_data_names]

nasdaq_prices = load_price_csv(raw_data_paths[0])
sp500_prices = load_price_csv(raw_data_paths[1])
nasdaq_prices.name = "NASDAQ"
sp500_prices.name = "S&P 500"

prices = pd.concat(
    {
        "NASDAQ": nasdaq_prices,
        "S&P 500": sp500_prices,
    },
    axis=1,
).dropna()
prices.index.name = "date"

prices.head(), prices.tail(), prices.shape

(                 NASDAQ     S&P 500
 date                               
 2009-04-15  1626.800049  852.059998
 2009-04-16  1670.439941  865.299988
 2009-04-17  1673.069946  869.599976
 2009-04-20  1608.209961  832.390015
 2009-04-21  1643.849976  850.080017,
                  NASDAQ      S&P 500
 date                                
 2015-10-06  4748.359863  1979.920044
 2015-10-07  4791.149902  1995.829956
 2015-10-08  4810.790039  2013.430054
 2015-10-09  4830.470215  2014.890015
 2015-10-12  4838.640137  2017.459961,
 (1636, 2))

In [ ]:
#prices.to_csv(PROCESSED_DATA_DIR / "prices_nasdaq_sp500.csv")

In [91]:
fig1 = plot_price_evolution(
    prices,
    output_path=REPORTS_DIR / "figures" / "figure_1_prices.html",
)

fig1.show()

In [ ]:
#fig1.write_image(REPORTS_DIR / "figures" / "figure_1_prices.png", scale=2)

## 2. Build returns

In [92]:
returns = align_return_frame(
    {
        "NASDAQ": 100 * log_returns(prices["NASDAQ"]),
        "S&P 500": 100 * log_returns(prices["S&P 500"]),
    }
)
returns.head(), returns.tail(), returns.shape

(              NASDAQ   S&P 500
 date                          
 2009-04-16  2.647210  1.541931
 2009-04-17  0.157320  0.495705
 2009-04-20 -3.953850 -4.373221
 2009-04-21  2.191930  2.102938
 2009-04-22  0.137996 -0.771132,
               NASDAQ   S&P 500
 date                          
 2015-10-06 -0.690479 -0.359469
 2015-10-07  0.897118  0.800352
 2015-10-08  0.409087  0.877978
 2015-10-09  0.408250  0.072485
 2015-10-12  0.168990  0.127466,
 (1635, 2))

In [ ]:
#returns.to_csv(PROCESSED_DATA_DIR / "returns_nasdaq_sp500.csv")

In [93]:
fig2 = plot_returns_and_squared_returns(
    returns,
    output_path=REPORTS_DIR / "figures" / "figure_2_returns_squared_returns.html",
)

fig2.show()

In [ ]:
# fig2.write_image(
#     REPORTS_DIR / "figures" / "figure_2_returns_squared_returns.png",
#     scale=2,
# )

## 3. Descriptive Statistics

In [94]:
stats = summary_statistics(returns)
stats

,count,Mean,Std,Skewness,Kurtosis,Hurst,Tail index,Arch(1),Arch(1) p-value,Arch(5),Arch(5) p-value,Arch(10),Arch(10) p-value,JB,JB p-value,ADF,ADF p-value
NASDAQ,1635.0,0.066668,1.139080,-0.393170,6.025142,-0.003873,3.852579,60.577994,7.072018e-15,243.084919,1.673343e-50,289.393210,2.708777e-56,660.180948,4.400770e-144,24.799764,0.0
S&P 500,1635.0,0.052718,1.035471,-0.429081,6.742166,-0.004462,3.615215,70.106714,5.618143e-17,304.898743,8.859261e-64,342.679085,1.424292e-67,996.403046,4.303558e-217,20.273255,0.0


In [95]:
table_1 = format_table_1(stats)
table_1

,Mean,Std,Skewness,Kurtosis,Hurst,Tail index,Arch(1),Arch(5),Arch(10),JB,ADF
NASDAQ,0.067,1.139,-0.393,6.025,-0.004,3.853,60.578\n(0.000),243.085\n(0.000),289.393\n(0.000),660.181\n(0.000),24.800\n(0.000)
S&P 500,0.053,1.035,-0.429,6.742,-0.004,3.615,70.107\n(0.000),304.899\n(0.000),342.679\n(0.000),996.403\n(0.000),20.273\n(0.000)


- DIFF HURST (papier NASDAQ: 0.436; SP500: 0.435)
- DIFF ADF (papier NASDAQ: 15.423)

In [ ]:
#table_1.to_csv(REPORTS_DIR / "tables" / "table_1_statistics.csv", index=True)

## 4. MSM

In [17]:
msm_table = fit_msm_grid(
    returns=returns,
    k_values=range(1, 8),
    n_starts=30,
    seed=123,
)
msm_table

Estimating MSM: asset=NASDAQ, k=1, n_starts=30
  start 1/30: x0=[1.5        1.13908017 2.         0.1       ]
    success=True, loglik=-2386.293, nit=9, nfev=70
  start 2/30: x0=[1.5        1.13908017 5.         0.2       ]
    success=True, loglik=-2386.293, nit=9, nfev=65
  start 3/30: x0=[ 1.4         1.13908017 10.          0.1       ]
    success=True, loglik=-2386.293, nit=10, nfev=65
  start 4/30: x0=[ 1.6         1.13908017 10.          0.2       ]
    success=True, loglik=-2386.293, nit=9, nfev=65
  start 5/30: x0=[ 1.3         1.13908017 20.          0.1       ]
    success=True, loglik=-2386.293, nit=11, nfev=70
  start 6/30: x0=[1.36645353 1.0272659  5.42513298 0.45135148]
    success=True, loglik=-2386.293, nit=14, nfev=135
  start 7/30: x0=[ 1.24072268  0.77442334 12.30695595  0.69617121]


KeyboardInterrupt: 

In [ ]:
loglik_pivot = msm_table.pivot(
    index="asset",
    columns="k",
    values="log_likelihood",
)

loglik_pivot

k,1,2,3,4,5,6,7
asset,,,,,,,
NASDAQ,-2386.292750,-2354.025698,-2353.841507,-2352.802364,-2352.691255,-2352.456376,-2352.649137
S&P 500,-2183.472284,-2136.246976,-2134.866108,-2135.584227,-2136.379933,-2137.191961,-2137.568090


In [ ]:
for param in ["m0", "sigma", "b", "gamma_k"]:
    display(
        msm_table.pivot(
            index="asset",
            columns="k",
            values=param,
        )
    )

k,1,2,3,4,5,6,7
asset,,,,,,,
NASDAQ,1.626379,1.549237,1.463521,1.493732,1.368557,1.333003,1.511452
S&P 500,1.701435,1.594094,1.548371,1.549645,1.547410,1.547560,1.503663


k,1,2,3,4,5,6,7
asset,,,,,,,
NASDAQ,1.304675,1.343376,1.281819,1.078176,1.277593,1.212689,5.587469
S&P 500,1.097379,1.227768,1.219545,0.980777,2.689507,0.632760,1.268974


k,1,2,3,4,5,6,7
asset,,,,,,,
NASDAQ,2.0,16.291868,5.004679,9.930332,2.424307,1.956567,20.565907
S&P 500,5.0,14.437770,19.975270,24.631064,19.772582,19.615558,3.479013


k,1,2,3,4,5,6,7
asset,,,,,,,
NASDAQ,0.040915,0.135141,0.274929,0.620093,0.202011,0.201380,0.906468
S&P 500,0.072633,0.119077,0.888056,0.931017,0.884964,0.882603,0.101175


In [ ]:
#msm_table.to_csv(REPORTS_DIR / "tables" / "table_2_msm_estimates.csv", index=False)

In [96]:
msm_table = pd.read_csv(REPORTS_DIR / "tables" / "table_2_msm_estimates.csv")

best k:
- NASDAQ = 3
- SP500 = 3

papier: SP500 best k = 4

### k=3 & k=4

we use the selected k in the paper

In [24]:
%%time

res_nasdaq_final = fit_msm(
    returns=returns["NASDAQ"],
    k=3,
    n_starts=30,
    seed=123,
)

res_sp500_final = fit_msm(
    returns=returns["S&P 500"],
    k=4,
    n_starts=30,
    seed=123,
)

  start 1/30: x0=[1.5        1.13908017 2.         0.1       ]
    success=True, loglik=-2355.671, nit=4, nfev=35
  start 2/30: x0=[1.5        1.13908017 5.         0.2       ]
    success=True, loglik=-2353.842, nit=8, nfev=65
  start 3/30: x0=[ 1.4         1.13908017 10.          0.1       ]
    success=True, loglik=-2355.813, nit=10, nfev=90
  start 4/30: x0=[ 1.6         1.13908017 10.          0.2       ]
    success=True, loglik=-2355.792, nit=6, nfev=60
  start 5/30: x0=[ 1.3         1.13908017 20.          0.1       ]
    success=True, loglik=-2354.930, nit=7, nfev=75
  start 6/30: x0=[1.5776463  0.74475455 7.54636434 0.19146578]
    success=True, loglik=-2354.246, nit=10, nfev=85
  start 7/30: x0=[ 1.22313413  1.60848884 27.79233594  0.27721419]
    success=True, loglik=-2355.302, nit=6, nfev=40
  start 8/30: x0=[ 1.67382819  1.69710722 15.97354911  0.24781708]
    success=True, loglik=-2354.873, nit=12, nfev=100
  start 9/30: x0=[ 1.67696912  0.92694125 22.5542511   0.6058443

In [25]:
msm_fit_results = {
    "NASDAQ": res_nasdaq_final,
    "S&P 500": res_sp500_final,
}

In [26]:
pit_msm = build_msm_pit_frame(
    returns=returns,
    fit_results=msm_fit_results,
)

pit_msm.head()

,NASDAQ,S&P 500
date,,
2009-04-16,0.968529,0.862457
2009-04-17,0.527258,0.635519
2009-04-20,0.011079,0.011669
2009-04-21,0.881390,0.838473
2009-04-22,0.517535,0.330242


In [ ]:
#pit_msm.to_csv(PROCESSED_DATA_DIR / "pit_msm.csv")

In [ ]:
pit_msm = pd.read_csv(
    PROCESSED_DATA_DIR / "pit_msm.csv",
    index_col="date",
    parse_dates=True,
)

### k=3

testing with empirical results

In [27]:
res_sp500_emp = fit_msm(
    returns=returns["S&P 500"],
    k=3,
    n_starts=30,
    seed=123,
)

  start 1/30: x0=[1.5        1.03547075 2.         0.1       ]
    success=True, loglik=-2139.144, nit=12, nfev=85
  start 2/30: x0=[1.5        1.03547075 5.         0.2       ]
    success=True, loglik=-2140.102, nit=7, nfev=60
  start 3/30: x0=[ 1.4         1.03547075 10.          0.1       ]
    success=True, loglik=-2136.720, nit=10, nfev=65
  start 4/30: x0=[ 1.6         1.03547075 10.          0.2       ]
    success=True, loglik=-2137.886, nit=7, nfev=65
  start 5/30: x0=[ 1.3         1.03547075 20.          0.1       ]
    success=True, loglik=-2134.866, nit=11, nfev=105
  start 6/30: x0=[1.5776463  0.67701254 7.54636434 0.19146578]
    success=True, loglik=-2136.834, nit=10, nfev=95
  start 7/30: x0=[ 1.22313413  1.46218256 27.79233594  0.27721419]
    success=True, loglik=-2137.798, nit=8, nfev=60
  start 8/30: x0=[ 1.67382819  1.54274031 15.97354911  0.24781708]
    success=True, loglik=-2137.092, nit=10, nfev=95
  start 9/30: x0=[ 1.67696912  0.84262775 22.5542511   0.60584

In [ ]:
msm_fit_results_emp = {
    "NASDAQ": res_nasdaq_final,
    "S&P 500": res_sp500_emp,
}
pit_msm_emp = build_msm_pit_frame(
    returns=returns,
    fit_results=msm_fit_results_emp,
)
#pit_msm_emp.to_csv(PROCESSED_DATA_DIR / "pit_msm_emp.csv")

In [83]:
pit_msm_emp = pd.read_csv(
    PROCESSED_DATA_DIR / "pit_msm_emp.csv",
    index_col="date",
    parse_dates=True,
)

### k=5

In [97]:
%%time

res_nasdaq_5 = fit_msm(
    returns=returns["NASDAQ"],
    k=5,
    n_starts=30,
    seed=123,
)

res_sp500_5 = fit_msm(
    returns=returns["S&P 500"],
    k=5,
    n_starts=30,
    seed=123,
)

  start 1/30: x0=[1.5        1.13908017 2.         0.1       ]
    success=True, loglik=-2352.691, nit=17, nfev=110
  start 2/30: x0=[1.5        1.13908017 5.         0.2       ]
    success=True, loglik=-2353.886, nit=10, nfev=80
  start 3/30: x0=[ 1.4         1.13908017 10.          0.1       ]
    success=True, loglik=-2355.485, nit=7, nfev=60
  start 4/30: x0=[ 1.6         1.13908017 10.          0.2       ]
    success=True, loglik=-2355.486, nit=7, nfev=50
  start 5/30: x0=[ 1.3         1.13908017 20.          0.1       ]
    success=True, loglik=-2355.321, nit=14, nfev=130
  start 6/30: x0=[1.5776463  0.74475455 7.54636434 0.19146578]
    success=True, loglik=-2358.019, nit=9, nfev=70
  start 7/30: x0=[ 1.22313413  1.60848884 27.79233594  0.27721419]
    success=True, loglik=-2355.565, nit=10, nfev=60
  start 8/30: x0=[ 1.67382819  1.69710722 15.97354911  0.24781708]
    success=True, loglik=-2350.959, nit=10, nfev=95
  start 9/30: x0=[ 1.67696912  0.92694125 22.5542511   0.6058

In [98]:
msm_fit_results_5 = {
    "NASDAQ": res_nasdaq_5,
    "S&P 500": res_sp500_5,
}

In [ ]:
pit_msm = build_msm_pit_frame(
    returns=returns,
    fit_results=msm_fit_results_5,
)
#pit_msm.to_csv(PROCESSED_DATA_DIR / "pit_msm_5.csv")

In [84]:
pit_msm_5 = pd.read_csv(
    PROCESSED_DATA_DIR / "pit_msm_5.csv",
    index_col="date",
    parse_dates=True,
)

## 5. GARCH

In [38]:
%%time

garch_results = fit_garch_marginals(
    returns=returns,
    mean="Constant",
    dist="normal",
    rescale=False,
)
garch_results

CPU times: total: 312 ms
Wall time: 557 ms


{'NASDAQ': GARCHFitResult(asset='NASDAQ', model_result=                     Constant Mean - GARCH Model Results                      
 Dep. Variable:                 NASDAQ   R-squared:                       0.000
 Mean Model:             Constant Mean   Adj. R-squared:                  0.000
 Vol Model:                      GARCH   Log-Likelihood:               -2368.82
 Distribution:                  Normal   AIC:                           4745.64
 Method:            Maximum Likelihood   BIC:                           4767.24
                                         No. Observations:                 1635
 Date:                Sat, May 09 2026   Df Residuals:                     1634
 Time:                        15:59:39   Df Model:                            1
                                 Mean Model                                
                  coef    std err          t      P>|t|    95.0% Conf. Int.
 -------------------------------------------------------------------------

In [39]:
garch_table = garch_results_table(garch_results)
garch_table

,asset,mean,mean_se,omega,omega_se,alpha,alpha_se,beta,beta_se,log_likelihood,...,Arch(5),Arch(5) p-value,Arch(10),Arch(10) p-value,Q(2),Q(2) p-value,Q(4),Q(4) p-value,Q(8),Q(8) p-value
0,NASDAQ,0.094821,0.023171,0.043995,0.011718,0.103481,0.020528,0.860258,0.024105,-2368.818971,...,8.185071,0.146327,10.492696,0.398381,0.037705,0.981324,3.775267,0.437274,4.959775,0.761867
1,S&P 500,0.073626,0.019626,0.034368,0.008225,0.125234,0.022874,0.841043,0.023353,-2161.970578,...,16.262379,0.006134,18.804659,0.042815,1.037366,0.595304,2.639331,0.619872,7.459761,0.487934


In [ ]:
#garch_table.to_csv(REPORTS_DIR / "tables" / "table_3_garch_estimates_raw.csv", index=False)

In [14]:
garch_table = pd.read_csv(REPORTS_DIR / "tables" / "table_3_garch_estimates_raw.csv")

In [41]:
table_3_garch = format_garch_table_3(garch_table)
table_3_garch

,omega,alpha,beta,Arch(1),Arch(5),Arch(10),Q(2),Q(4),Q(8)
asset,,,,,,,,,
NASDAQ,0.044 [0.012],0.103 [0.021],0.860 [0.024],1.740 (0.187),8.185 (0.146),10.493 (0.398),0.038 (0.981),3.775 (0.437),4.960 (0.762)
S&P 500,0.034 [0.008],0.125 [0.023],0.841 [0.023],4.222 (0.040),16.262 (0.006),18.805 (0.043),1.037 (0.595),2.639 (0.620),7.460 (0.488)


In [ ]:
#table_3_garch.to_csv(REPORTS_DIR / "tables" / "table_3_garch_estimates_formatted.csv")

In [16]:
table_3_garch = pd.read_csv(REPORTS_DIR / "tables" / "table_3_garch_estimates_formatted.csv")

### Conditional Vol

In [43]:
garch_volatility = build_garch_volatility_frame(garch_results)
garch_volatility

,NASDAQ,S&P 500
date,,
2009-04-16,1.771182,1.768811
2009-04-17,1.848471,1.713396
2009-04-20,1.727358,1.589261
2009-04-21,2.075341,2.152921
2009-04-22,2.050428,2.109120
...,...,...
2015-10-06,1.413941,1.298467
2015-10-07,1.351911,1.214854
2015-10-08,1.297253,1.158351


In [ ]:
#garch_volatility.to_csv(PROCESSED_DATA_DIR / "garch_conditional_volatility.csv")

In [99]:
garch_volatility = pd.read_csv(PROCESSED_DATA_DIR / "garch_conditional_volatility.csv", index_col="date", parse_dates=True)

### PIT GARCH

In [45]:
pit_garch = build_garch_pit_frame(garch_results)
pit_garch

,NASDAQ,S&P 500
date,,
2009-04-16,0.925217,0.796761
2009-04-17,0.513486,0.597291
2009-04-20,0.009543,0.002571
2009-04-21,0.843870,0.827054
2009-04-22,0.508400,0.344385
...,...,...
2015-10-06,0.289311,0.369362
2015-10-07,0.723561,0.725147
2015-10-08,0.595709,0.756282


In [46]:
pit_garch.describe()

,NASDAQ,S&P 500
count,1635.000000,1635.000000
mean,0.499149,0.498485
std,0.275217,0.273879
min,0.000041,0.000022
25%,0.301117,0.291500
50%,0.506885,0.499895
75%,0.720656,0.714563
max,0.997744,0.999363


In [47]:
pit_garch.isna().sum()

NASDAQ     0
S&P 500    0
dtype: int64

In [ ]:
#pit_garch.to_csv(PROCESSED_DATA_DIR / "pit_garch.csv")

In [85]:
pit_garch = pd.read_csv(
    PROCESSED_DATA_DIR / "pit_garch.csv",
    index_col="date",
    parse_dates=True,
)

## 6. Copulas

### MSM specifications used below

We keep three MSM specifications:

1. `paper_like_table2`: NASDAQ \(k=3\), S&P 500 \(k=4\), close to the model selection in Table 2.
2. `empirical_best`: NASDAQ \(k=3\), S&P 500 \(k=3\), selected from our likelihood table.
3. `paper_like_var`: NASDAQ \(k=5\), S&P 500 \(k=5\), because Table 4 and the VaR section of Segnon and Trede fix the MSM volatility components at \(k=5\).

The main VaR backtest will start with the `paper_like_var` specification.

In [19]:
pit_msm = pd.read_csv(
    PROCESSED_DATA_DIR / "pit_msm_5.csv",
    index_col="date",
    parse_dates=True,
)
pit_msm_emp = pd.read_csv(
    PROCESSED_DATA_DIR / "pit_msm_emp.csv",
    index_col="date",
    parse_dates=True,
)
pit_garch = pd.read_csv(
    PROCESSED_DATA_DIR / "pit_garch.csv",
    index_col="date",
    parse_dates=True,
)

### paperilike : k=5

In [86]:
%%time

copula_results = fit_copula_grid(
    pit_by_model={
        "MSM": pit_msm,
        "GARCH": pit_garch,
    }
)

c:\Users\alixg\OneDrive - Université Paris-Dauphine\M2 Quant\Gestion Quantitative\Réplication 2\Forecasting_market_risk-Copula_MSM_Approach\src\copulas.py:181: RuntimeWarning: divide by zero encountered in divide
  
c:\Users\alixg\OneDrive - Université Paris-Dauphine\M2 Quant\Gestion Quantitative\Réplication 2\Forecasting_market_risk-Copula_MSM_Approach\src\copulas.py:181: RuntimeWarning: divide by zero encountered in divide
  
c:\Users\alixg\OneDrive - Université Paris-Dauphine\M2 Quant\Gestion Quantitative\Réplication 2\Forecasting_market_risk-Copula_MSM_Approach\src\copulas.py:181: RuntimeWarning: divide by zero encountered in divide
  
c:\Users\alixg\OneDrive - Université Paris-Dauphine\M2 Quant\Gestion Quantitative\Réplication 2\Forecasting_market_risk-Copula_MSM_Approach\src\copulas.py:181: RuntimeWarning: divide by zero encountered in divide
  
c:\Users\alixg\OneDrive - Université Paris-Dauphine\M2 Quant\Gestion Quantitative\Réplication 2\Forecasting_market_risk-Copula_MSM_Appro

CPU times: total: 9.34 s
Wall time: 9.73 s


In [87]:
copula_table = copula_results_table(copula_results)
copula_table

,margin_model,copula,log_likelihood,aic,bic,nobs,success,message,rho,nu,theta
0,MSM,gaussian,1790.237713,-3578.475426,-3573.076028,1635,True,CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*...,0.945020,NaN,NaN
1,MSM,student,1816.377223,-3628.754447,-3617.955651,1635,True,CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*...,0.944521,5.628161,NaN
2,MSM,plackett,1681.462603,-3360.925206,-3355.525808,1635,True,CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*...,NaN,NaN,75.222744
3,MSM,clayton,1471.764013,-2941.528025,-2936.128627,1635,True,CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*...,NaN,NaN,4.612738
4,MSM,rotated_clayton,1505.239404,-3008.478808,-3003.079410,1635,True,CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*...,NaN,NaN,4.888786
5,MSM,frank,1532.348929,-3062.697858,-3057.298460,1635,True,CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*...,NaN,NaN,20.000000
6,MSM,gumbel,1758.985979,-3515.971958,-3510.572560,1635,True,CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*...,NaN,NaN,4.480543
7,MSM,rotated_gumbel,1728.424141,-3454.848283,-3449.448885,1635,True,CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*...,NaN,NaN,4.389183
8,GARCH,gaussian,1841.477032,-3680.954064,-3675.554666,1635,True,CONVERGENCE: NORM OF PROJECTED GRADIENT <= PGTOL,0.945868,NaN,NaN
9,GARCH,student,1846.797074,-3689.594149,-3678.795353,1635,True,CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*...,0.946423,17.532837,NaN


In [88]:
table_4 = format_copula_table_4(copula_table)
table_4

,margin_model,copula,parameters,Log(L),AIC,BIC
0,MSM,gaussian,rho=0.945,1790.238,-3578.475,-3573.076
1,MSM,student,"rho=0.945, nu=5.628",1816.377,-3628.754,-3617.956
2,MSM,plackett,theta=75.223,1681.463,-3360.925,-3355.526
3,MSM,clayton,theta=4.613,1471.764,-2941.528,-2936.129
4,MSM,rotated_clayton,theta=4.889,1505.239,-3008.479,-3003.079
5,MSM,frank,theta=20.000,1532.349,-3062.698,-3057.298
6,MSM,gumbel,theta=4.481,1758.986,-3515.972,-3510.573
7,MSM,rotated_gumbel,theta=4.389,1728.424,-3454.848,-3449.449
8,GARCH,gaussian,rho=0.946,1841.477,-3680.954,-3675.555
9,GARCH,student,"rho=0.946, nu=17.533",1846.797,-3689.594,-3678.795


In [ ]:
# copula_table.to_csv(
#     REPORTS_DIR / "tables" / "table_4_copula_estimates_raw.csv",
#     index=False,
# )

In [ ]:
# table_4.to_csv(
#     REPORTS_DIR / "tables" / "table_4_copula_estimates_formatted.csv",
#     index=False,
# )

In [80]:
best_loglik = (
    copula_table
    .loc[copula_table.groupby("margin_model")["log_likelihood"].idxmax()]
    .sort_values("margin_model")
)
best_aic = (
    copula_table
    .loc[copula_table.groupby("margin_model")["aic"].idxmin()]
    .sort_values("margin_model")
)
best_bic = (
    copula_table
    .loc[copula_table.groupby("margin_model")["bic"].idxmin()]
    .sort_values("margin_model")
)

best_summary = pd.concat(
    {
        "Max Log(L)": best_loglik,
        "Min AIC": best_aic,
        "Min BIC": best_bic,
    },
    names=["criterion"]
).reset_index(level=0)
best_summary[
    ["criterion", "margin_model", "copula", "log_likelihood", "aic", "bic", "rho", "nu", "theta"]
]

,criterion,margin_model,copula,log_likelihood,aic,bic,rho,nu,theta
9,Max Log(L),GARCH,student,1846.797074,-3689.594149,-3678.795353,0.946422,17.532720,NaN
1,Max Log(L),MSM,student,1816.377223,-3628.754447,-3617.955651,0.944521,5.628161,NaN
9,Min AIC,GARCH,student,1846.797074,-3689.594149,-3678.795353,0.946422,17.532720,NaN
1,Min AIC,MSM,student,1816.377223,-3628.754447,-3617.955651,0.944521,5.628161,NaN
9,Min BIC,GARCH,student,1846.797074,-3689.594149,-3678.795353,0.946422,17.532720,NaN
1,Min BIC,MSM,student,1816.377223,-3628.754447,-3617.955651,0.944521,5.628161,NaN


### empirical_best: k=3

In [76]:
%%time

copula_results_emp = fit_copula_grid(
    pit_by_model={
        "MSM": pit_msm_emp,
        "GARCH": pit_garch,
    }
)

c:\Users\alixg\OneDrive - Université Paris-Dauphine\M2 Quant\Gestion Quantitative\Réplication 2\Forecasting_market_risk-Copula_MSM_Approach\src\copulas.py:182: RuntimeWarning: divide by zero encountered in divide
  density = numerator / denominator
c:\Users\alixg\OneDrive - Université Paris-Dauphine\M2 Quant\Gestion Quantitative\Réplication 2\Forecasting_market_risk-Copula_MSM_Approach\src\copulas.py:182: RuntimeWarning: divide by zero encountered in divide
  density = numerator / denominator
c:\Users\alixg\OneDrive - Université Paris-Dauphine\M2 Quant\Gestion Quantitative\Réplication 2\Forecasting_market_risk-Copula_MSM_Approach\src\copulas.py:182: RuntimeWarning: divide by zero encountered in divide
  density = numerator / denominator
c:\Users\alixg\OneDrive - Université Paris-Dauphine\M2 Quant\Gestion Quantitative\Réplication 2\Forecasting_market_risk-Copula_MSM_Approach\src\copulas.py:182: RuntimeWarning: divide by zero encountered in divide
  density = numerator / denominator
c:\U

CPU times: total: 5.36 s
Wall time: 5.49 s


In [77]:
copula_table_emp = copula_results_table(copula_results_emp)
table_4_emp = format_copula_table_4(copula_table_emp)
table_4_emp

,margin_model,copula,parameters,Log(L),AIC,BIC
0,MSM,gaussian,rho=0.946,1799.563,-3597.125,-3591.726
1,MSM,student,"rho=0.945, nu=5.576",1825.919,-3647.837,-3637.038
2,MSM,plackett,theta=75.673,1686.451,-3370.902,-3365.503
3,MSM,clayton,theta=4.657,1485.549,-2969.098,-2963.699
4,MSM,rotated_clayton,theta=4.903,1510.657,-3019.315,-3013.915
5,MSM,frank,theta=20.000,1537.477,-3072.953,-3067.554
6,MSM,gumbel,theta=4.494,1766.015,-3530.029,-3524.630
7,MSM,rotated_gumbel,theta=4.415,1740.900,-3479.799,-3474.400
8,GARCH,gaussian,rho=0.946,1841.477,-3680.954,-3675.555
9,GARCH,student,"rho=0.946, nu=17.533",1846.797,-3689.594,-3678.795


In [ ]:
# copula_table_emp.to_csv(
#     REPORTS_DIR / "tables" / "table_4_copula_estimates_raw_emp.csv",
#     index=False,
# )

In [ ]:
# table_4_emp.to_csv(
#     REPORTS_DIR / "tables" / "table_4_copula_estimates_formatted_emp.csv",
#     index=False,
# )

In [81]:
best_loglik_emp = (
    copula_table_emp
    .loc[copula_table_emp.groupby("margin_model")["log_likelihood"].idxmax()]
    .sort_values("margin_model")
)
best_aic_emp = (
    copula_table_emp
    .loc[copula_table_emp.groupby("margin_model")["aic"].idxmin()]
    .sort_values("margin_model")
)
best_bic_emp = (
    copula_table_emp
    .loc[copula_table_emp.groupby("margin_model")["bic"].idxmin()]
    .sort_values("margin_model")
)

best_summary = pd.concat(
    {
        "Max Log(L)": best_loglik_emp,
        "Min AIC": best_aic_emp,
        "Min BIC": best_bic_emp,
    },
    names=["criterion"]
).reset_index(level=0)
best_summary[
    ["criterion", "margin_model", "copula", "log_likelihood", "aic", "bic", "rho", "nu", "theta"]
]

,criterion,margin_model,copula,log_likelihood,aic,bic,rho,nu,theta
9,Max Log(L),GARCH,student,1846.797074,-3689.594149,-3678.795353,0.946422,17.532720,NaN
1,Max Log(L),MSM,student,1825.918610,-3647.837221,-3637.038425,0.944905,5.575925,NaN
9,Min AIC,GARCH,student,1846.797074,-3689.594149,-3678.795353,0.946422,17.532720,NaN
1,Min AIC,MSM,student,1825.918610,-3647.837221,-3637.038425,0.944905,5.575925,NaN
9,Min BIC,GARCH,student,1846.797074,-3689.594149,-3678.795353,0.946422,17.532720,NaN
1,Min BIC,MSM,student,1825.918610,-3647.837221,-3637.038425,0.944905,5.575925,NaN


## 7. Portfolio VaR

In [139]:
import importlib
import src.var as var_module

importlib.reload(var_module)

from src.var import (
    forecast_msm_copula_var_oos_fixed_params, 
    forecast_historical_var, 
    forecast_riskmetrics_var, 
    forecast_variance_covariance_var, 
    forecast_ccc_garch_var_fixed_params, 
    forecast_garch_copula_var_fixed_params,
)

### Important methodological note

The paper uses a rolling estimation scheme: the first 1135 observations are used for estimation, the last 500 observations are used for out-of-sample evaluation, and the estimation window is rolled forward one day at a time.

In this notebook, the first VaR implementation is a fixed-parameter approximation: the MSM marginal parameters and the Student copula parameters are fixed, and only the Hamilton filter probabilities vary over time. This is faster and useful as a first validation step, but it is not yet the full rolling-window replication of the paper.

### 7.1 Parameters

In [133]:
PI = 0.5
WEIGHTS = np.array([PI, 1.0 - PI])
N_OOS = 500
WINDOW_SIZE = 1135

#### Student copula GARCH

In [134]:
student_garch_row = copula_table.query(
    "margin_model == 'GARCH' and copula == 'student'"
).iloc[0]

COPULA_GARCH_PARAMS = {
    "rho": float(student_garch_row["rho"]),
    "nu": float(student_garch_row["nu"]),
}

COPULA_GARCH_PARAMS

{'rho': 0.9464226620481486, 'nu': 17.53283692470736}

#### Student Copula MSM

In [135]:
student_msm_row = copula_table.query(
    "margin_model == 'MSM' and copula == 'student'"
).iloc[0]

COPULA = "student"
COPULA_PARAMS = {
    "rho": float(student_msm_row["rho"]),
    "nu": float(student_msm_row["nu"]),
}


In [136]:
# Paper-like VaR uses k=5 for both MSM marginals
MSM_1_VAR = res_nasdaq_5
MSM_2_VAR = res_sp500_5
r1 = returns["NASDAQ"]
r2 = returns["S&P 500"]

t_start = len(returns) - N_OOS
oos_index = returns.index[t_start:]

COPULA_PARAMS, oos_index[0], oos_index[-1]

({'rho': 0.9445206983906996, 'nu': 5.628160778464574},
 Timestamp('2013-10-17 00:00:00'),
 Timestamp('2015-10-12 00:00:00'))

### 7.2 VaR Forecast 5% and 1%

- Historical, covariance et RiskMetrics sont bien rolling.
- CCC-GARCH, Copula-GARCH et Copula-MSM sont ici fixed-parameter.
- Le papier utilise un vrai rolling scheme pour l’évaluation complète.

#### 5%

In [121]:
%%time

var_5pct = forecast_msm_copula_var_oos_fixed_params(
    msm_1=MSM_1_VAR,
    msm_2=MSM_2_VAR,
    returns_1=r1,
    returns_2=r2,
    copula_params=COPULA_PARAMS,
    copula=COPULA,
    pi=PI,
    alpha=0.05,
    n_oos=N_OOS,
    integration_nodes=501,
    root_tol=1e-4,
    verbose=True,
)

var_5pct.describe()

  [50/500] VaR 5.0% = -1.1374
  [100/500] VaR 5.0% = -1.1071
  [150/500] VaR 5.0% = -1.3268
  [200/500] VaR 5.0% = -1.4038
  [250/500] VaR 5.0% = -1.9731
  [300/500] VaR 5.0% = -1.5069
  [350/500] VaR 5.0% = -1.1708
  [400/500] VaR 5.0% = -1.1516
  [450/500] VaR 5.0% = -1.3627
  [500/500] VaR 5.0% = -2.0024
VaR 5% — mean=-1.3889, min=-3.0862, max=-0.9488
CPU times: total: 26min 37s
Wall time: 19min 50s


date
2013-10-17   -1.645022
2013-10-18   -1.587119
2013-10-21   -1.613536
2013-10-22   -1.501192
2013-10-23   -1.431028
Name: VaR_5_return, dtype: float64

In [122]:
var_5pct.describe()

count    500.000000
mean      -1.388899
std        0.387610
min       -3.086235
25%       -1.540455
50%       -1.282832
75%       -1.116759
max       -0.948780
Name: VaR_5_return, dtype: float64

In [ ]:
var_5pct.to_csv(DATA_DIR / "processed" / "var_5pct_student_copula_msm.csv", index=True)

In [140]:
%%time

var_5_historical = forecast_historical_var(
    returns=returns[["NASDAQ", "S&P 500"]],
    alpha=0.05,
    weights=WEIGHTS,
    window_size=WINDOW_SIZE,
    n_oos=N_OOS,
)

var_5_riskmetrics = forecast_riskmetrics_var(
    returns=returns[["NASDAQ", "S&P 500"]],
    alpha=0.05,
    weights=WEIGHTS,
    lambda_=0.94,
    window_size=WINDOW_SIZE,
    n_oos=N_OOS,
    include_mean=False,
)

var_5_covariance = forecast_variance_covariance_var(
    returns=returns[["NASDAQ", "S&P 500"]],
    alpha=0.05,
    weights=WEIGHTS,
    window_size=WINDOW_SIZE,
    n_oos=N_OOS,
    include_mean=True,
)

var_5_ccc_garch = forecast_ccc_garch_var_fixed_params(
    returns=returns[["NASDAQ", "S&P 500"]],
    garch_results=garch_results,
    alpha=0.05,
    weights=WEIGHTS,
    n_oos=N_OOS,
    include_mean=True,
)

var_5_copula_garch = forecast_garch_copula_var_fixed_params(
    returns=returns[["NASDAQ", "S&P 500"]],
    garch_results=garch_results,
    copula_params=COPULA_GARCH_PARAMS,
    copula="student",
    alpha=0.05,
    weights=WEIGHTS,
    n_oos=N_OOS,
    integration_nodes=501,
    root_tol=1e-4,
)

CPU times: total: 18min 3s
Wall time: 10min 15s


In [141]:
var_5_all = pd.concat(
    [
        var_5_historical.rename("Historical"),
        var_5_riskmetrics.rename("RiskMetrics"),
        var_5_covariance.rename("Covariance"),
        var_5_ccc_garch.rename("CCC-GARCH"),
        var_5_copula_garch.rename("Student-Copula-GARCH"),
        var_5pct.rename("Student-Copula-MSM"),
    ],
    axis=1,
).dropna(how="any")

var_5_all.describe()

,Historical,RiskMetrics,Covariance,CCC-GARCH,Student-Copula-GARCH,Student-Copula-MSM
count,500.000000,500.000000,500.000000,500.000000,500.000000,500.000000
mean,-1.724853,-1.361241,-1.682472,-1.360795,-1.360842,-1.388899
std,0.070753,0.417182,0.070559,0.412640,0.412651,0.387610
min,-1.894145,-3.012619,-1.828738,-3.506524,-3.506634,-3.086235
25%,-1.774394,-1.513424,-1.745814,-1.503053,-1.503104,-1.540455
50%,-1.729161,-1.261211,-1.703078,-1.246831,-1.246881,-1.282832
75%,-1.650264,-1.107127,-1.600546,-1.101133,-1.101185,-1.116759
max,-1.637277,-0.797875,-1.585432,-0.888690,-0.888727,-0.948780


In [143]:
var_5_all.to_csv(PROCESSED_DATA_DIR / "var_5pct_all_models.csv", index=True)

In [144]:
portfolio_oos_5 = portfolio_returns(
    returns=returns[["NASDAQ", "S&P 500"]],
    weights=WEIGHTS,
).loc[var_5_all.index]

In [145]:
fig3 = plot_var_forecasts(
    portfolio_returns=portfolio_oos_5.rename("Portfolio returns"),
    var_forecasts=var_5_all,
    output_path=REPORTS_DIR / "figures" / "figure_3_var_5pct_all_models.html",
    title="Figure 3 — VaR forecasts at the 5% confidence level",
    positive_loss_var=False,
)

fig3.show()

In [146]:
fig3.write_image(
    REPORTS_DIR / "figures" / "figure_3_var_5pct_all_models.png",
    scale=2,
)

#### 1%

In [123]:
%%time

var_1pct = forecast_msm_copula_var_oos_fixed_params(
    msm_1=MSM_1_VAR,
    msm_2=MSM_2_VAR,
    returns_1=r1,
    returns_2=r2,
    copula_params=COPULA_PARAMS,
    copula=COPULA,
    pi=PI,
    alpha=0.01,
    n_oos=N_OOS,
    integration_nodes=501,
    root_tol=1e-4,
    verbose=True,
)

print(
    f"VaR 1% — mean={var_1pct.mean():.4f}, "
    f"min={var_1pct.min():.4f}, max={var_1pct.max():.4f}"
)

var_1pct.head()

  [50/500] VaR 1.0% = -1.9563
  [100/500] VaR 1.0% = -1.8876
  [150/500] VaR 1.0% = -2.2630
  [200/500] VaR 1.0% = -2.3729
  [250/500] VaR 1.0% = -3.3079
  [300/500] VaR 1.0% = -2.5223
  [350/500] VaR 1.0% = -2.0155
  [400/500] VaR 1.0% = -1.9830
  [450/500] VaR 1.0% = -2.3178
  [500/500] VaR 1.0% = -3.4050
VaR 1% — mean=-2.3340, min=-4.8337, max=-1.6248
CPU times: total: 27min 35s
Wall time: 20min 28s


date
2013-10-17   -2.666674
2013-10-18   -2.595903
2013-10-21   -2.627489
2013-10-22   -2.494913
2013-10-23   -2.406931
Name: VaR_1_return, dtype: float64

In [124]:
var_1pct.describe()

count    500.000000
mean      -2.333954
std        0.610680
min       -4.833671
25%       -2.552140
50%       -2.195280
75%       -1.913344
max       -1.624842
Name: VaR_1_return, dtype: float64

In [147]:
%%time

var_1_historical = forecast_historical_var(
    returns=returns[["NASDAQ", "S&P 500"]],
    alpha=0.01,
    weights=WEIGHTS,
    window_size=WINDOW_SIZE,
    n_oos=N_OOS,
)

var_1_riskmetrics = forecast_riskmetrics_var(
    returns=returns[["NASDAQ", "S&P 500"]],
    alpha=0.01,
    weights=WEIGHTS,
    lambda_=0.94,
    window_size=WINDOW_SIZE,
    n_oos=N_OOS,
    include_mean=False,
)

var_1_covariance = forecast_variance_covariance_var(
    returns=returns[["NASDAQ", "S&P 500"]],
    alpha=0.01,
    weights=WEIGHTS,
    window_size=WINDOW_SIZE,
    n_oos=N_OOS,
    include_mean=True,
)

var_1_ccc_garch = forecast_ccc_garch_var_fixed_params(
    returns=returns[["NASDAQ", "S&P 500"]],
    garch_results=garch_results,
    alpha=0.01,
    weights=WEIGHTS,
    n_oos=N_OOS,
    include_mean=True,
)

var_1_copula_garch = forecast_garch_copula_var_fixed_params(
    returns=returns[["NASDAQ", "S&P 500"]],
    garch_results=garch_results,
    copula_params=COPULA_GARCH_PARAMS,
    copula="student",
    alpha=0.01,
    weights=WEIGHTS,
    n_oos=N_OOS,
    integration_nodes=501,
    root_tol=1e-4,
)

CPU times: total: 24min 29s
Wall time: 14min 34s


In [148]:
var_1_all = pd.concat(
    [
        var_1_historical.rename("Historical"),
        var_1_riskmetrics.rename("RiskMetrics"),
        var_1_covariance.rename("Covariance"),
        var_1_ccc_garch.rename("CCC-GARCH"),
        var_1_copula_garch.rename("Student-Copula-GARCH"),
        var_1pct.rename("Student-Copula-MSM"),
    ],
    axis=1,
).dropna(how="any")
var_1_all.to_csv(PROCESSED_DATA_DIR / "var_1pct_all_models.csv", index=True)

portfolio_oos_1 = portfolio_returns(
    returns=returns[["NASDAQ", "S&P 500"]],
    weights=WEIGHTS,
).loc[var_1_all.index]

fig4 = plot_var_forecasts(
    portfolio_returns=portfolio_oos_1.rename("Portfolio returns"),
    var_forecasts=var_1_all,
    output_path=REPORTS_DIR / "figures" / "figure_4_var_1pct_all_models.html",
    title="Figure 4 — VaR forecasts at the 1% confidence level",
    positive_loss_var=False,
)

fig4.show()

fig4.write_image(
    REPORTS_DIR / "figures" / "figure_4_var_1pct_all_models.png",
    scale=2,
)

### 7.3 Save forecasts

In [125]:
var_forecasts = pd.DataFrame(
    {
        "Student-Copula-MSM VaR 5%": var_5pct,
        "Student-Copula-MSM VaR 1%": var_1pct,
    }
)
var_forecasts.to_csv(PROCESSED_DATA_DIR / "var_forecasts_copula_msm_student_k5.csv", index=True)
var_forecasts.head()

,Student-Copula-MSM VaR 5%,Student-Copula-MSM VaR 1%
date,,
2013-10-17,-1.645022,-2.666674
2013-10-18,-1.587119,-2.595903
2013-10-21,-1.613536,-2.627489
2013-10-22,-1.501192,-2.494913
2013-10-23,-1.431028,-2.406931


### 7.4 Figures

In [ ]:
portfolio_oos = portfolio_returns(
    returns=returns[["NASDAQ", "S&P 500"]],
    weights=np.array(WEIGHTS),
).loc[var_forecasts.index]

fig3 = plot_var_forecasts(
    portfolio_returns=portfolio_oos,
    var_forecasts=var_forecasts,
    output_path=REPORTS_DIR / "figures" / "figure_3_var_copula_msm_student_k5.html",
    title="VaR forecasts — Student-Copula-MSM",
    positive_loss_var=False,
)

fig3.show()

In [127]:
fig3.write_image(
    REPORTS_DIR / "figures" / "figure_3_var_copula_msm_student_k5.png",
    scale=2,
)

### Pre-backtest

In [129]:
hits_5pct = var_exceedances(
    returns=portfolio_oos,
    var_forecasts=var_forecasts["Student-Copula-MSM VaR 5%"],
)
hits_1pct = var_exceedances(
    returns=portfolio_oos,
    var_forecasts=var_forecasts["Student-Copula-MSM VaR 1%"],
)

pre_backtest_summary = pd.DataFrame(
    {
        "alpha": [0.05, 0.01],
        "nobs": [len(hits_5pct), len(hits_1pct)],
        "violations": [hits_5pct.sum(), hits_1pct.sum()],
        "efv": [hits_5pct.mean(), hits_1pct.mean()],
    },
    index=["VaR 5%", "VaR 1%"],
)

pre_backtest_summary

,alpha,nobs,violations,efv
VaR 5%,0.05,500,35,0.070
VaR 1%,0.01,500,6,0.012
